# Cohere Transcribe WER evaluation on FLEURS

This notebook measures sample WER locally on Apple Silicon. 
It loads the official Cohere BF16 checkpoint once through `mlx-audio`, then evaluates a deterministic sample from the requested FLEURS language configuration.

## 1. Import dependencies and locate the repository

This cell imports the dataset, Hugging Face, MLX, evaluation, and reporting libraries. 

It also locates the repository root so that the notebook can import the local WER helpers regardless of the directory from which JupyterLab was started. Run it first.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from datasets import Audio, load_dataset
from huggingface_hub import hf_hub_download
from IPython.display import display
from mlx_audio.stt.utils import load_model

repository_root = Path.cwd().resolve()
while not (repository_root / "macos" / "wer-test" / "wer_utils.py").is_file():
    if repository_root.parent == repository_root:
        raise RuntimeError(
            "Start Jupyter from the cohere-voice repository or a child directory."
        )
    repository_root = repository_root.parent

sys.path.insert(0, str(repository_root / "macos" / "wer-test"))
from wer_utils import (
    calculate_wer,
    get_fleurs_config,
    iter_indexed_batches,
    normalize_transcript,
    validate_positive,
)

## 2. Configure the evaluation

Set the model, language, FLEURS split, sample size, random seed, and MLX batch size here. 

The default evaluates a deterministic sample of 100 **Italian** records from the `test` split. 
Change these values before downloading data or loading the model.

In [2]:
# Editable evaluation settings
MODEL_ID = "CohereLabs/cohere-transcribe-03-2026"
DATASET_ID = "google/fleurs"

# the language !!
LANGUAGE = "it"

SPLIT = "test"
SAMPLE_SIZE = 100
SEED = 42
BATCH_SIZE = 1
MAX_TOKENS = 256

validate_positive(SAMPLE_SIZE, "Sample size")
validate_positive(BATCH_SIZE, "Batch size")
DATASET_CONFIG = get_fleurs_config(LANGUAGE)
DATASET_FILENAME = f"parquet-data/{DATASET_CONFIG}/{SPLIT}-00000-of-00001.parquet"
OUTPUT_DIR = repository_root / "macos" / "wer-test" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET_ID} / {DATASET_CONFIG} / {SPLIT}")
print(
    f"Language: {LANGUAGE}; sample size: {SAMPLE_SIZE}; seed: {SEED}; batch size: {BATCH_SIZE}"
)
print(f"Dataset file: {DATASET_FILENAME}")

Dataset: google/fleurs / it_it / test
Language: it; sample size: 100; seed: 42; batch size: 1
Dataset file: parquet-data/it_it/test-00000-of-00001.parquet


## 3. Download and sample the selected FLEURS shard

This cell downloads the Parquet shard for the selected language and split. Hugging Face shows byte-level progress on the first download and reuses the local cache later. 

The shard is then loaded locally, shuffled with the configured seed, and reduced to the requested sample size.

In [3]:
# Download exactly one language-and-split Parquet shard. The Hugging Face
# client shows byte-level progress during the first download and reuses its cache
# on future runs.
print(f"Downloading selected FLEURS shard: {DATASET_FILENAME}")
dataset_path = hf_hub_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    filename=DATASET_FILENAME,
)

dataset = load_dataset(
    "parquet", data_files={SPLIT: str(dataset_path)}, split=SPLIT
).cast_column("audio", Audio(decode=False))
if SAMPLE_SIZE > len(dataset):
    raise RuntimeError(
        f"Requested {SAMPLE_SIZE} samples but split has only {len(dataset)}."
    )

samples = dataset.shuffle(seed=SEED).select(range(SAMPLE_SIZE))
print(f"Loaded {len(samples)} deterministic shuffled samples from {dataset_path}.")

Loaded 100 deterministic shuffled samples from /Users/lsaetta/.cache/huggingface/hub/datasets--google--fleurs/snapshots/70bb2e84b976b7e960aa89f1c648e09c59f894dd/parquet-data/it_it/test-00000-of-00001.parquet.


## 4. Prepare local audio and transcription helpers

FLEURS stores audio bytes inside the Parquet file. These helpers materialize each selected clip under the ignored output directory, then pass local file paths to MLX Audio. 

Batch errors fall back to one-file-at-a-time inference so that individual failures are recorded instead of aborting the whole evaluation.

In [4]:
def materialize_audio(sample: dict, audio_dir: Path) -> Path:
    """Write embedded Parquet audio bytes to a local file for MLX Audio."""
    audio = sample["audio"]
    source_path = Path(audio.get("path") or "")
    if source_path.is_file():
        return source_path

    audio_bytes = audio.get("bytes")
    if audio_bytes is None:
        raise ValueError(
            f"Dataset record {sample['id']} has no accessible audio bytes."
        )

    suffix = source_path.suffix or ".wav"
    audio_path = audio_dir / f"{sample['id']}{suffix}"
    audio_path.write_bytes(audio_bytes)
    return audio_path


def transcribe_batch(batch: list[dict], model, audio_dir: Path) -> list[dict]:
    """Transcribe local audio files, recording individual errors if a batch fails."""
    audio_paths = [str(materialize_audio(sample, audio_dir)) for sample in batch]
    try:
        hypotheses = model.transcribe(
            audio_files=audio_paths,
            language=LANGUAGE,
            batch_size=BATCH_SIZE,
            max_tokens=MAX_TOKENS,
        )
        return [{"hypothesis": text, "error": None} for text in hypotheses]
    except Exception as batch_error:
        outcomes = []
        for audio_path in audio_paths:
            try:
                output = model.generate(
                    audio_path, language=LANGUAGE, max_tokens=MAX_TOKENS
                )
                outcomes.append({"hypothesis": output.text, "error": None})
            except Exception as sample_error:
                outcomes.append(
                    {
                        "hypothesis": None,
                        "error": f"{type(batch_error).__name__}: {batch_error}; fallback: {type(sample_error).__name__}: {sample_error}",
                    }
                )
        return outcomes

## 5. Run local Cohere Transcribe inference

This is the GPU-intensive cell. 

It loads the official Cohere checkpoint once with `mlx-audio`, processes the selected records through MLX and Metal, and retains the raw and normalized reference/hypothesis pairs needed for scoring. Audio remains on the local machine after the dataset download.

In [5]:
# Load the MLX model once. This requires Apple Silicon with Metal available.
model = load_model(MODEL_ID)
run_name = datetime.now(timezone.utc).strftime("wer_%Y%m%dT%H%M%SZ")
audio_dir = OUTPUT_DIR / f"{run_name}_audio"
audio_dir.mkdir(parents=True, exist_ok=True)
rows = []

for start, batch in iter_indexed_batches(samples, BATCH_SIZE):
    outcomes = transcribe_batch(batch, model, audio_dir)
    for offset, (sample, outcome) in enumerate(zip(batch, outcomes)):
        reference = sample["transcription"]
        hypothesis = outcome["hypothesis"]
        normalized_reference = normalize_transcript(reference)
        normalized_hypothesis = normalize_transcript(hypothesis) if hypothesis else None
        sample_wer = (
            calculate_wer([normalized_reference], [normalized_hypothesis])
            if normalized_hypothesis is not None
            else None
        )
        rows.append(
            {
                "sample_index": start + offset,
                "dataset_id": sample["id"],
                "reference": reference,
                "hypothesis": hypothesis,
                "normalized_reference": normalized_reference,
                "normalized_hypothesis": normalized_hypothesis,
                "sample_wer": sample_wer,
                "error": outcome["error"],
            }
        )

    print(f"Completed {min(start + BATCH_SIZE, len(samples))}/{len(samples)} samples.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Completed 1/100 samples.
Completed 2/100 samples.
Completed 3/100 samples.
Completed 4/100 samples.
Completed 5/100 samples.
Completed 6/100 samples.
Completed 7/100 samples.
Completed 8/100 samples.
Completed 9/100 samples.
Completed 10/100 samples.
Completed 11/100 samples.
Completed 12/100 samples.
Completed 13/100 samples.
Completed 14/100 samples.
Completed 15/100 samples.
Completed 16/100 samples.
Completed 17/100 samples.
Completed 18/100 samples.
Completed 19/100 samples.
Completed 20/100 samples.
Completed 21/100 samples.
Completed 22/100 samples.
Completed 23/100 samples.
Completed 24/100 samples.
Completed 25/100 samples.
Completed 26/100 samples.
Completed 27/100 samples.
Completed 28/100 samples.
Completed 29/100 samples.
Completed 30/100 samples.
Completed 31/100 samples.
Completed 32/100 samples.
Completed 33/100 samples.
Completed 34/100 samples.
Completed 35/100 samples.
Completed 36/100 samples.
Completed 37/100 samples.
Completed 38/100 samples.
Completed 39/100 samp

## 6. Calculate and save WER results

This final cell computes corpus WER using only successful transcriptions, displays the aggregate summary and a preview of detailed rows, and saves auditable CSV and JSON files under `macos/wer-test/output/`. 

Lower WER is better; `0.0` is an exact match after the documented text normalization.

In [6]:
results = pd.DataFrame(rows)
successful = results[results["error"].isna()].copy()
if successful.empty:
    raise RuntimeError("No successful transcripts are available for WER calculation.")

corpus_wer = calculate_wer(
    successful["normalized_reference"].tolist(),
    successful["normalized_hypothesis"].tolist(),
)
summary = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "split": SPLIT,
    "language": LANGUAGE,
    "requested_samples": SAMPLE_SIZE,
    "successful_samples": len(successful),
    "failed_samples": int(results["error"].notna().sum()),
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "normalization": "NFKC, lowercase, punctuation removal, whitespace collapse",
    "corpus_wer": corpus_wer,
}

results_path = OUTPUT_DIR / f"{run_name}_details.csv"
summary_path = OUTPUT_DIR / f"{run_name}_summary.json"
results.to_csv(results_path, index=False)
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

display(pd.DataFrame([summary]))
display(
    results[["sample_index", "reference", "hypothesis", "sample_wer", "error"]].head(10)
)
print(f"Detailed results: {results_path}")
print(f"Summary: {summary_path}")

,timestamp_utc,model_id,dataset_id,dataset_config,split,language,requested_samples,successful_samples,failed_samples,seed,batch_size,normalization,corpus_wer
0,2026-08-18T16:24:03.693519+00:00,CohereLabs/cohere-transcribe-03-2026,google/fleurs,it_it,test,it,100,100,0,42,1,"NFKC, lowercase, punctuation removal, whitespa...",0.027205


,sample_index,reference,hypothesis,sample_wer,error
0,0,vista la topologia sottomarina il flusso di ri...,"Vista la topologia sottomarina, il flusso di r...",0.000000,None
1,1,se attraversate il mar baltico settentrionale ...,Se attraversate il Mar Baltico settentrionale ...,0.000000,None
2,2,infine esistono molti piccoli felini compresi ...,Infine esistono molti piccoli felini compresi ...,0.000000,None
3,3,lo shock da rientro si manifesta in un tempo p...,Lo shock da rientro si manifesta in un tempo p...,0.000000,None
4,4,non è stata emanata alcuna allerta tsunami e s...,Non è stata emanata alcuna allerta tsunami e s...,0.029412,None
5,5,usa gymnastics e usoc hanno il medesimo obiett...,USA Gymnastics e Yussock hanno il medesimo obi...,0.058824,None
6,6,gli scienziati hanno potuto desumere che gli e...,Gli scienziati hanno potuto desumere che gli e...,0.000000,None
7,7,l'iniziativa contro le oscenità è stata finanz...,L'iniziativa contro le ostilità è stata finanz...,0.066667,None
8,8,tuttavia ghiaccio e neve sono condizioni norma...,"Tuttavia, ghiaccio e neve sono condizioni norm...",0.000000,None
9,9,è una delle attrazioni principali del sudafric...,È una delle attrazioni principali del Sudafric...,0.111111,None


Detailed results: /Users/lsaetta/Progetti/cohere-voice/macos/wer-test/output/wer_20260818T162314Z_details.csv
Summary: /Users/lsaetta/Progetti/cohere-voice/macos/wer-test/output/wer_20260818T162314Z_summary.json
